# 05 · Calibration and threshold selection
### Calibration design
- Class weights inflate predicted PD, and the 2015 base rate differs from the training base rate, so **level** matters as much as ranking.
- **Deployable recalibration** = Platt scaling fitted on **train out-of-fold** scores. Fitting it on in-sample scores would be optimistic (especially for GBM); fitting it on validation would be leakage.
- **ORACLE intercept shift** (uses validation outcomes) is shown only as a **diagnostic**: it tells us how much of any remaining gap is pure base-rate shift vs. a shape problem. It is not a deployable model.
- Per-decile **binomial tests** are reported; with large n they flag even small gaps, so read the *size* of the gap (rel_gap), not just the p-value.

### Threshold design
**Not F1.** F1 treats a false positive and a false negative as symmetric and ignores exposure size, LGD and risk appetite. We pick the cut-off from an **expected-loss vs. approval-rate trade-off**: approve the lowest-PD share of applicants; EL = PD x LGD_proxy x CCF_proxy x loan_amnt. Choose the **largest approval rate whose approved-book EL rate stays within an assumed appetite** (config input; not a bank policy).

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.utils.config import load_config, set_seeds
cfg = load_config(); set_seeds(cfg["seed"])
pd.set_option("display.float_format", "{:.4f}".format); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
print("config seed:", cfg["seed"], "| raw file:", cfg["paths"]["raw_csv"])

In [ ]:
from src.pipeline import prepare, get_models, calibration_tables
from src.evaluation.metrics import calibration_by_decile
data = prepare(cfg); models = get_models(data, cfg)
cal = calibration_tables(models, data)
cal["champion_logit"]["summary"]

In [ ]:
c = cal["champion_logit"]
print("Decile calibration, train-only Platt (deployable):"); display(c["decile_cal"])
print("Decile calibration, raw class-weighted (before recalibration):"); display(c["decile_raw"])

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, n in zip(axs, ["champion_logit","challenger_lgbm"]):
    for lab, key in [("raw","decile_raw"),("train-only Platt","decile_cal"),("ORACLE (diag.)","decile_oracle")]:
        t = cal[n][key]; ax.plot(t.expected_pd, t.observed_dr, "o-", label=lab)
    m = cal[n]["decile_raw"][["expected_pd","observed_dr"]].values.max(); ax.plot([0,m],[0,m],"k:"); ax.set(title=n, xlabel="expected PD", ylabel="observed default rate"); ax.legend()
plt.tight_layout(); plt.show()
cal["challenger_lgbm"]["summary"]

**2016 (flagged, partially censored)** - reported separately; do not read its calibration as clean evidence.

In [ ]:
y16 = data.flagged.target.astype(int); p16 = models["champion_logit"].predict(data.flagged)
from src.evaluation.metrics import calibration_summary
pd.Series(calibration_summary(y16, p16)).to_frame("champion on 2016 (censored)")

### Trade-off table
LGD and CCF come from **train defaults only** (notebook 06 details them) and are **proxies**.

In [ ]:
from src.models.el_proxy import fit_lgd_ead_proxy
from src.decision.threshold import tradeoff_table, recommend, ks_optimal_cutoff
el = fit_lgd_ead_proxy(data.train[data.train.target==1]); print(el)
v = data.valid.assign(pd_hat=models["champion_logit"].predict(data.valid))
trade = tradeoff_table(v, "pd_hat", el["lgd"], el["ccf"], cfg["decision"]["approval_grid"], cfg["decision"]["el_appetite"])
trade

In [ ]:
rec = recommend(trade); ksc = ks_optimal_cutoff(v.target.astype(int), v.pd_hat)
print(f"Assumed EL appetite: {cfg['decision']['el_appetite']:.1%}")
print(f"Recommended: approve {rec.approval_rate:.0%} (PD cutoff {rec.pd_cutoff:.3f}); approved-book model EL rate {rec.el_rate_model:.2%}; defaults rejected {rec.defaults_rejected_share:.0%}; good loans rejected {rec.good_loans_rejected_share:.0%}")
print(f"KS-optimal PD cutoff (reference only, ignores exposure/LGD/appetite): {ksc:.3f}  -> approval rate at that cutoff: {(v.pd_hat<=ksc).mean():.0%}")
fig, ax = plt.subplots(figsize=(6.5,4.2))
ax.plot(trade.approval_rate*100, trade.el_rate_model*100, "o-", label="model EL rate"); ax.plot(trade.approval_rate*100, trade.loss_rate_realised_proxy*100, "s--", label="realised (proxy LGD/EAD)")
ax.axhline(cfg["decision"]["el_appetite"]*100, color="r", ls=":", label="assumed appetite"); ax.set(xlabel="approval rate (%)", ylabel="loss rate on approved exposure (%)"); ax.legend(); plt.show()
(ROOT/cfg["paths"]["reports_dir"]).mkdir(exist_ok=True, parents=True); trade.to_csv(ROOT/cfg["paths"]["reports_dir"]/"threshold_tradeoff.csv", index=False)

**Reading the table.** Each step down in approval rate rejects a lot of defaults but also many good loans; the cost of a rejected good loan (foregone margin) is **not** modelled here, which is why the recommendation is conditional on a stated appetite rather than presented as "optimal". Changing `decision.el_appetite` in `configs/config.yaml` moves the answer.

**Hosmer-Lemeshow calibration test.**

In [ ]:
from scipy import stats
for n in ['champion_logit', 'challenger_lgbm']:
    t = cal[n]['decile_cal']
    hl = ((t['defaults'] - t['n']*t['expected_pd'])**2 /
          (t['n']*t['expected_pd']*(1-t['expected_pd']))).sum()
    df = len(t) - 2
    print(f'{n}: HL chi2 = {hl:.2f}, p = {1 - stats.chi2.cdf(hl, df):.4f}')